# Monthly Broker Prediction Script

Use this notebook to make predictions on new monthly data using the trained three-stage pipeline.

## Prerequisites
1. Run `01_broker_classification_pipeline.ipynb` first to train the models
2. Ensure saved models exist in `../data/models/`
3. Prepare new month's data in the same format as training data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
from datetime import datetime

print("Libraries imported successfully!")
print(f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Load Trained Models

In [ ]:
# Load the trained models
with open('../data/models/stage1_model.pkl', 'rb') as f:
    model_s1 = pickle.load(f)

with open('../data/models/stage2_model.pkl', 'rb') as f:
    model_s2 = pickle.load(f)

with open('../data/models/stage3_model.pkl', 'rb') as f:
    model_s3 = pickle.load(f)

with open('../data/models/feature_columns.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

print("✅ Models loaded successfully!")
print(f"Number of features: {len(feature_cols)}")

## 2. Load New Month's Data

**INSTRUCTIONS:**
- Place the new month's Excel file in `../data/raw/`
- Update the filename below
- The data should have the same structure as the training data

In [ ]:
# Load new month's data
# UPDATE THIS FILENAME:
new_data_file = '../data/raw/new_month_data.xlsx'  # Change this to your new file

try:
    new_data = pd.read_excel(new_data_file)
    print(f"✅ Data loaded successfully!")
    print(f"Shape: {new_data.shape}")
    print(f"\nColumns: {new_data.columns.tolist()[:5]}...")  # Show first 5 columns
    print(f"\nFirst few rows:")
    display(new_data.head())
except FileNotFoundError:
    print(f"❌ File not found: {new_data_file}")
    print("Please update the filename and ensure the file is in the correct location.")

## 3. Validate Data Structure

In [ ]:
# Check if all required features are present
missing_features = [col for col in feature_cols if col not in new_data.columns]

if missing_features:
    print(f"⚠️  WARNING: Missing {len(missing_features)} features:")
    for feat in missing_features[:10]:  # Show first 10
        print(f"   - {feat}")
    print("\n⚠️  Predictions may be inaccurate. Ensure data format matches training data.")
else:
    print("✅ All required features are present!")

# Check for missing values
missing_vals = new_data[feature_cols].isnull().sum().sum()
if missing_vals > 0:
    print(f"\n⚠️  Found {missing_vals} missing values. Will fill with 0.")
else:
    print("✅ No missing values found!")

## 4. Run Three-Stage Prediction Pipeline

In [ ]:
def predict_broker_classes_pipeline(data, feature_cols, model_s1, model_s2, model_s3):
    """
    Complete three-stage prediction pipeline.
    
    Stage 1: Predict Others vs nonOthers
    Stage 2: Confirm nonOthers classification
    Stage 3: Classify into Whale/Dolphin/Minnow
    """
    results = data.copy()
    
    # Prepare features
    X = results[feature_cols].fillna(0)
    
    print("🔄 Stage 1: Predicting Others vs nonOthers...")
    # Stage 1: Others vs nonOthers
    stage1_pred = model_s1.predict(X)
    stage1_proba = model_s1.predict_proba(X)
    results['Stage1_Prediction'] = stage1_pred
    
    # Get probability for nonOthers class
    nonothers_idx = list(model_s1.classes_).index('nonOthers')
    results['Stage1_Confidence'] = stage1_proba[:, nonothers_idx]
    
    stage1_others = (stage1_pred == 'Others').sum()
    stage1_nonothers = (stage1_pred == 'nonOthers').sum()
    print(f"   ✓ Others: {stage1_others}, nonOthers: {stage1_nonothers}")
    
    # Stage 2: Confirm nonOthers
    print("\n🔄 Stage 2: Confirming nonOthers...")
    nonothers_mask = stage1_pred == 'nonOthers'
    results['Stage2_Prediction'] = 'N/A (Predicted Others)'
    results['Stage2_Confidence'] = np.nan
    
    if nonothers_mask.sum() > 0:
        stage2_pred = model_s2.predict(X[nonothers_mask])
        stage2_proba = model_s2.predict_proba(X[nonothers_mask])
        
        results.loc[nonothers_mask, 'Stage2_Prediction'] = stage2_pred
        
        nonothers_idx_s2 = list(model_s2.classes_).index('nonOthers')
        results.loc[nonothers_mask, 'Stage2_Confidence'] = stage2_proba[:, nonothers_idx_s2]
        
        confirmed = (stage2_pred == 'nonOthers').sum()
        rejected = (stage2_pred == 'Others').sum()
        print(f"   ✓ Confirmed nonOthers: {confirmed}, Rejected: {rejected}")
    else:
        print("   ⚠️  No nonOthers to confirm")
    
    # Stage 3: Classify into W/D/M
    print("\n🔄 Stage 3: Classifying into Whale/Dolphin/Minnow...")
    confirmed_mask = (results['Stage1_Prediction'] == 'nonOthers') & \
                     (results['Stage2_Prediction'] == 'nonOthers')
    
    results['Final_Class'] = 'Others'
    results['Stage3_Confidence'] = np.nan
    
    if confirmed_mask.sum() > 0:
        stage3_pred = model_s3.predict(X[confirmed_mask])
        stage3_proba = model_s3.predict_proba(X[confirmed_mask])
        
        results.loc[confirmed_mask, 'Final_Class'] = stage3_pred
        
        # Get max probability for each prediction
        results.loc[confirmed_mask, 'Stage3_Confidence'] = stage3_proba.max(axis=1)
        
        whales = (stage3_pred == 'Whale').sum()
        dolphins = (stage3_pred == 'Dolphin').sum()
        minnows = (stage3_pred == 'Minnow').sum()
        print(f"   ✓ Whales: {whales}, Dolphins: {dolphins}, Minnows: {minnows}")
    else:
        print("   ⚠️  No confirmed nonOthers to classify")
    
    # Handle rejected nonOthers
    rejected_mask = (results['Stage1_Prediction'] == 'nonOthers') & \
                    (results['Stage2_Prediction'] == 'Others')
    results.loc[rejected_mask, 'Final_Class'] = 'Others (Rejected in Stage 2)'
    
    return results

print("✅ Prediction function ready!")

In [ ]:
# Run the prediction pipeline
print("="*70)
print("RUNNING THREE-STAGE PREDICTION PIPELINE")
print("="*70)

predictions = predict_broker_classes_pipeline(
    new_data,
    feature_cols,
    model_s1,
    model_s2,
    model_s3
)

print("\n" + "="*70)
print("✅ PREDICTIONS COMPLETE!")
print("="*70)

## 5. View Results

In [ ]:
# Summary statistics
print("PREDICTION SUMMARY")
print("="*70)
print(f"\nTotal Brokers: {len(predictions)}")
print(f"\n📊 Final Class Distribution:")
print(predictions['Final_Class'].value_counts())
print(f"\n📊 Final Class Percentages:")
print(predictions['Final_Class'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# View detailed results
output_cols = ['Account Name', 'Owner Name', 'Stage1_Prediction', 'Stage1_Confidence',
               'Stage2_Prediction', 'Stage2_Confidence', 'Final_Class', 'Stage3_Confidence']

results_df = predictions[output_cols].copy()
results_df = results_df.round(3)

print("Sample Results (first 20 brokers):")
display(results_df.head(20))

In [ ]:
# View by class
print("\n" + "="*70)
print("WHALES (High Priority - 6+ competitive loans expected)")
print("="*70)
whales = results_df[results_df['Final_Class'] == 'Whale']
display(whales.sort_values('Stage3_Confidence', ascending=False))

print("\n" + "="*70)
print("DOLPHINS (Medium Priority - 4-5 competitive loans expected)")
print("="*70)
dolphins = results_df[results_df['Final_Class'] == 'Dolphin']
display(dolphins.sort_values('Stage3_Confidence', ascending=False))

print("\n" + "="*70)
print("MINNOWS (Lower Priority - 2-3 competitive loans expected)")
print("="*70)
minnows = results_df[results_df['Final_Class'] == 'Minnow']
display(minnows.sort_values('Stage3_Confidence', ascending=False))

## 6. Export Results

In [ ]:
# Export to Excel
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f'../data/processed/broker_predictions_{timestamp}.xlsx'

# Create a writer with multiple sheets
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Full results
    predictions.to_excel(writer, sheet_name='All_Predictions', index=False)
    
    # Summary by class
    whales = predictions[predictions['Final_Class'] == 'Whale']
    dolphins = predictions[predictions['Final_Class'] == 'Dolphin']
    minnows = predictions[predictions['Final_Class'] == 'Minnow']
    others = predictions[predictions['Final_Class'].str.contains('Others')]
    
    if len(whales) > 0:
        whales.to_excel(writer, sheet_name='Whales', index=False)
    if len(dolphins) > 0:
        dolphins.to_excel(writer, sheet_name='Dolphins', index=False)
    if len(minnows) > 0:
        minnows.to_excel(writer, sheet_name='Minnows', index=False)
    if len(others) > 0:
        others.to_excel(writer, sheet_name='Others', index=False)

print(f"✅ Results exported successfully!")
print(f"📁 File location: {output_file}")
print(f"\n📊 Export includes:")
print(f"   - All predictions")
print(f"   - Whales: {len(whales)}")
print(f"   - Dolphins: {len(dolphins)}")
print(f"   - Minnows: {len(minnows)}")
print(f"   - Others: {len(others)}")

## Notes for Your Boss

### How to Interpret the Results:

1. **Stage1_Confidence**: How confident the model is that the broker will be nonOthers (higher = more confident)
2. **Stage2_Confidence**: Confirmation confidence for nonOthers classification
3. **Stage3_Confidence**: Confidence in the Whale/Dolphin/Minnow classification

### Final Classes:
- **Whale**: Expected 6+ competitive loans next month (HIGH PRIORITY)
- **Dolphin**: Expected 4-5 competitive loans next month (MEDIUM PRIORITY)
- **Minnow**: Expected 2-3 competitive loans next month (LOWER PRIORITY)
- **Others**: Expected < 2 competitive loans next month (MINIMAL FOCUS)

### Action Items:
1. Focus retention efforts on Whales and Dolphins
2. Review brokers with low confidence scores - may need manual review
3. Compare predictions with actual results next month to validate model performance